### Astrogator Scripting Tool
this code snippet will 
- connect to a running instance of STK
- create a new satellite
- build a simple MCS
- set up the scripting tool

author: jens ramrath\
date: 21 april 2026

In [103]:
# connect to running instance of STK
from agi.stk13.stkdesktop import STKDesktop
from agi.stk13.stkobjects import *

stk = STKDesktop.AttachToApplication()

# Get the IAgStkObjectRoot interface
root = stk.Root
sc = root.CurrentScenario
print('connected to ' + sc.InstanceName)

connected to 20260421_13


In [104]:
# create satellite
if sc.Children.Contains(AgESTKObjectType.eSatellite, 'ScriptingToolSat'):
    sat = root.GetObjectFromPath('Satellite/ScriptingToolSat')
    sat.Unload()

sat = sc.Children.New(AgESTKObjectType.eSatellite, 'ScriptingToolSat')

In [105]:
# build simple MCS
sat.SetPropagatorType(AgEVePropagatorType.ePropagatorAstrogator)
prop = sat.Propagator
mcs = prop.MainSequence
mcs.RemoveAll()

### add segments
# initial state
iState = mcs.InsertByName('Initial_State', '-')
# prop 90 min
prop1 = mcs.InsertByName('Propagate', '-')
prop1.StoppingConditions['Duration'].Properties.Trip = 5400
# target sequence
ts = mcs.InsertByName('Target_Sequence', '-')
# maneuver
man = ts.Segments.InsertByName('Maneuver', '-')
# prop to apo
prop2 = ts.Segments.InsertByName('Propagate', '-')
prop2.StoppingConditions.Add('Apoapsis')
prop2.StoppingConditions.Remove('Duration')

prop.RunMCS()

In [106]:
# add initial guess scripting to MCS
dc = ts.Profiles.GetItemByIndex(0)
st = ts.Profiles.Add2('Scripting Tool', 0, 0)
st.Enable = True
# add object property
st.SegmentProperties.RemoveAll()
dvGuess = st.SegmentProperties.Add('DV_Guess')
dvGuess.ObjectName = 'Maneuver'
dvGuess.Attribute = 'ImpulsiveMnvr.Pointing.Cartesian.X'
# add calc object
st.CalcObjects.RemoveAll()
sma = st.CalcObjects.Add('sma')
sma.CalcObject.Name = 'Semimajor Axis'
sma.CalcObjectName = 'Semimajor Axis'
sma.ComponentName = 'sma'
mu = st.CalcObjects.Add('mu')
mu.CalcObject.Name = 'Gravitational Parameter'
mu.CalcObjectName = 'Gravitational Parameter'
mu.ComponentName = 'mu'
# simple initial guess script
st.ScriptText('v0 = Math.sqrt(mu/sma);\r\nvt = Math.sqrt(2*mu/sma - mu/((42166 + sma)/2));\r\nDV_Guess = vt - v0;')

prop.RunMCS()

In [107]:
# set up target sequence
man.EnableControlParameter(408) #eVAControlManeuverImpulsiveCartesianX
dvX = dc.ControlParameters.GetControlByPaths("Maneuver", "ImpulsiveMnvr.Cartesian.X")
dvX.Enable = True

prop2.Results.Add("Spherical Elems/R Mag")
rMag = dc.Results.GetResultByPaths("Propagate", "R Mag")
rMag.Enable = True
rMag.DesiredValue = 42166

ts.Action = 1  # eVATargetSeqActionRunActiveProfiles

prop.RunMCS()